# Using the C# Model Context Protocol SDK

A fast-paced, demo-focused course covering how to build, secure, and deploy MCP servers using the C# SDK. Topics include creating remote (Streamable HTTP) and local (STDIO) servers, wiring them to APIs, adding OAuth security, consuming them from AI agents, and publishing developer-targeted servers to NuGet.org.

## Table of Contents

- [Module 01: Getting Started with the MCP SDK](#module-01)
- [Module 02: Creating an MCP Server for APIs](#module-02)
- [Module 03: Security Considerations for MCP Servers](#module-03)
- [Module 04: Using an MCP Server from AI Agents](#module-04)
- [Module 05: Creating Developer-Targeted MCP Servers and Deploying to NuGet.org](#module-05)

---
## Module 01: Getting Started with the MCP SDK

_Introduces the C# MCP SDK, the demo application (CarvedRock), and the key MCP feature primitives available in the SDK._

### System Requirements

- **.NET SDK** — required runtime and toolchain
- **IDE of choice** — Visual Studio, VS Code, or Rider
- **Aspire** — used for orchestrating multi-service demos
- **Docker Desktop or equivalent** — Rancher or Podman also work

### The C# MCP SDK

The SDK ships as two NuGet packages:

- **`ModelContextProtocol`** — core SDK for all server types
- **`ModelContextProtocol.AspNetCore`** — integration for ASP.NET Core (Streamable HTTP transport)

Both are in pre-release status but are fully usable. API documentation is available on Microsoft Learn under *AI Tools and SDKs*.

Source: https://github.com/modelcontextprotocol/csharp-sdk/

### MCP Features in the SDK

| Feature | Purpose |
|---|---|
| **Tools** | Execute API calls, DB queries, or perform computations |
| **Resources** | Provide context (files, DB schemas, etc.) via URI |
| **Prompts** | Provide templated/standardised prompts (slash commands) |

**Transports:**
- **STDIO** — for local (in-process or sidecar) servers
- **Streamable HTTP** — for remote servers

**Security:** authentication and authorisation are first-class concerns, addressed through ASP.NET Core middleware and OAuth.

### Demo Application: CarvedRock

CarvedRock is a fictional fitness equipment company used throughout the course. The goal is to add AI features to its existing UI using MCP:

- MCP server makes API calls to the CarvedRock backend
- Proper security is enforced at the MCP layer
- Both e-commerce (user-facing) and admin features are covered

The starting-point project (`carved-rock-starting-point`) contains the full Aspire-hosted application — API, web app, data layer, and domain — before any MCP integration.

### Demo: Getting Started — Hello MCP and CarvedRock Starting Point

Two demo projects are provided for this module:

- **`hello-csharp-mcp-sdk/`** — a minimal "hello world" MCP server with both an `Hello.AspNetCoreMcp` (Streamable HTTP) and `Hello.Mcp` (STDIO) project, illustrating the simplest possible server setup.
- **`carved-rock-starting-point/`** — the full CarvedRock application (Aspire AppHost + API + WebApp + data/domain layers) used as the baseline before MCP tools are added.

See [`01-getting-started-demos/`](01-getting-started-demos/) for the full projects.

---
## Module 02: Creating an MCP Server for APIs

_Covers how to expose API methods as MCP tools, the different strategies for registering tools, and what logic belongs inside a tool._

### Feature Goal

**Feature request:** Provide product recommendations in a chat interface.

- User inputs what they are looking for
- AI uses that input plus the product list to make recommendations

**Our tasks:**
1. Create an MCP server with a tool that returns all products
2. Create automated tests for the tool

> **Remember:** Too many tools can confuse an AI service. Keep tool surface area intentional and minimal.

### Adding Tools to an MCP Server

The SDK offers four strategies for registering tools:

**`WithToolsFromAssembly`**
- Scans the provided assembly (or the calling assembly) at startup
- Uses reflection to find classes annotated with `[McpServerToolType]` and methods annotated with `[McpServerTool]`

**`WithTools<Type>`**
- Registers `[McpServerTool]` methods from a single, explicitly named class

**`WithTools(IEnumerable<McpServerTool>)`**
- Pass a manually constructed list of tools — useful when you need full control

**Dynamically via `ConfigureSessionOptions`**
- Decide which tools to expose per session — for example, based on the caller's authorisation claims

### What Logic Goes in an MCP Tool?

A tool is not a simple pass-through — it should add value over the raw API:

- **Cache content** — pre-fetch or hold data so the AI doesn't hammer the backend
- **Computed fields** — derive values the API doesn't return directly
- **Shape content** — trim, rename, or restructure responses to reduce token noise
- **Combine API methods** — merge multiple endpoints into one cohesive tool
- **Simplify API call for common use case** — hide rarely-needed parameters
- **Custom logic** — any business logic that bridges the gap between API and AI intent

> **Always consider the output (and input!) of tool calls.** Poorly shaped data wastes tokens and degrades reasoning.

### Feature Goal (Part 2)

**Feature request:** Enable AI to help admins update product prices and remove products.

**Our tasks:**
1. Add tools for the update-price and delete-product API methods
2. Create automated tests for these tools

### Demo: Creating an MCP Server for the CarvedRock API

This module builds the first real MCP server on top of the CarvedRock API. The demo shows adding product-listing, price-update, and delete tools with automated tests.

See [`02-api-server-demos/`](02-api-server-demos/) for the full project (begin and end states).

---
## Module 03: Security Considerations for MCP Servers

_Covers how to secure a Streamable HTTP MCP server using ASP.NET Core OAuth, with four distinct security scenarios and practical steps for enabling role-based access._

### Key Security Questions

Before implementing security, three questions frame the design:

- **Should listing tools or connecting be allowed anonymously?**
- **Should every tool be secured?**
- **Do some tools require different permissions?**

Answering these determines your security architecture.

### OAuth as the Security Standard

For Streamable HTTP MCP servers, **OAuth is the preferred way to implement security**.

Reference: https://modelcontextprotocol.io/docs/tutorials/security/authorization

The good news: **ASP.NET Core security techniques can be used directly** — no MCP-specific security framework needed.

### Four Security Scenarios to Consider

| Scenario | Description | OAuth Grant Type |
|---|---|---|
| **Anonymous access** | No security — open to anyone | — |
| **Machine-to-machine** | AI doing background work without a user | `client_credentials` |
| **User-based access** | Simplest is "logged in"; can be role-based, policy-based, or custom | `authorization_code` (interactive) |
| **AI on behalf of logged-in user** | AI acts using the user's delegated identity | Token exchange (delegated) |

### What This Module Covers (Demo Tasks)

1. Enable OAuth for the MCP server and test it with the MCP Inspector
2. Forward tokens for downstream API requests
3. Refine logging and telemetry — logs should include user identity and tool call information
4. Implement role-based access for admin-only operations (delete product, update price)
5. Create automated tests for secured tools

### Related Knowledge Areas

The OAuth and telemetry patterns shown in this module draw on:

- Authentication and Authorization in ASP.NET Core
- Securing ASP.NET Core with OAuth2 and OpenID Connect
- Logging and Monitoring in ASP.NET Core
- Configuration in ASP.NET Core

The demo moves quickly through these — deeper coverage of each area lives in dedicated resources.

### Demo: Securing the CarvedRock MCP Server

This module adds OAuth to the MCP server, wires up token forwarding to the API, adds structured logging with user context, and enforces role-based access on admin tools.

See [`03-security-demos/`](03-security-demos/) for the full project (begin and end states).

---
## Module 04: Using an MCP Server from AI Agents

_Shows how to build an AI agent that consumes MCP tools, including slash commands backed by MCP Prompts, token forwarding, token exchange, and the MCP elicitation feature._

### What This Module Covers

1. **Create an API method for an AI agent** — start with "hello, world" AI style, then improve setup and logging
2. **Add MCP tools** — the agent calls the secured CarvedRock MCP server for product recommendations
3. **Admin functionality via slash command** — backed by an MCP Prompt, requires token forwarding
4. **Token exchange sample** — demonstrates how the agent forwards a user token to the MCP server
5. **Elicitation feature** — the MCP protocol feature that allows a server to request additional input from the client mid-call

### Demo: Building an AI Agent on Top of the MCP Server

This module wires up a .NET AI agent to the secured MCP server, demonstrates slash-command prompts, and explores the elicitation protocol feature.

See [`04-agent-demos/`](04-agent-demos/) for the full project (begin and end states).

---
## Module 05: Creating Developer-Targeted MCP Servers and Deploying to NuGet.org

_Covers a different server archetype: an MCP server aimed at developers (not end users), created from the Microsoft.Extensions.AI template, configured for IDE integration, and published to NuGet.org._

### What This Module Covers

1. Create a new developer-targeted MCP server using the `Microsoft.Extensions.AI.Templates` template
2. Try it in VS Code via `.vscode/mcp.json`
3. Add a test data generation tool
4. Create server metadata in `.mcp/server.json` and in the `.csproj` file
5. Pack and publish to NuGet.org
6. Run the published package directly from NuGet

### Adding an MCP Server to GitHub Copilot

MCP servers can be registered with different IDEs and at different scopes:

| IDE | File Location | Scope of Server |
|---|---|---|
| VS Code | `WORKSPACEDIR/.vscode/mcp.json` | Available only in workspace |
| VS Code | `settings.json` (appdata roaming) | Available globally |
| Visual Studio | `SOLUTIONDIR/.mcp.json` | Available only in solution |
| Visual Studio | `%USERPROFILE%/.mcp.json` | Available globally |

### What Logic Goes in an MCP Tool? (Recap)

This applies equally to developer-targeted servers — a tool should add value over the raw underlying operation:

- **Cache content** — avoid repeated expensive lookups
- **Computed fields** — derive values the underlying source doesn't expose
- **Shape content** — return only what the AI needs, reducing token usage
- **Combine API methods** — merge multiple calls into one tool
- **Simplify API call for common use case** — hide rarely-needed parameters
- **Custom logic** — any glue code that makes the tool genuinely useful

> **Always consider the output (and input!) of tool calls.**

### Demo: CarvedRock Developer MCP Server

Two demo projects are provided for this module:

- **`carvedrock-application/`** — the full CarvedRock application (end state with all MCP integrations) used as the application-side reference
- **`carvedrock-developer/`** — a standalone developer-targeted MCP server with a `Tools/` directory, `Program.cs` entry point, and project metadata — ready to pack and publish to NuGet.org

See [`05-developer-server-demos/`](05-developer-server-demos/) for the full projects.

---

> Confidently enable AI features for your applications!